# Machine Learning Assignment 2
## Multi-Model Classification on the Online Shoppers Purchasing Intention Dataset

**Programme:** M.Tech (AIML) - Work Integrated Learning Programmes Division, BITS Pilani
**Course:** Machine Learning
**Environment:** BITS Virtual Lab

---

### Problem statement
Predict whether an e-commerce browsing session ends in a purchase (`Revenue = True`) from
session behaviour, page-value analytics and visitor attributes. This is a **binary
classification** problem with a **class imbalance** of roughly 85:15.

### What this notebook does
1. Loads and profiles the dataset
2. Builds a leakage-safe preprocessing pipeline (fit on train only)
3. Trains 5 classifiers on the same feature set
4. Scores each on Accuracy, AUC, Precision, Recall, F1 and MCC
5. Persists every fitted pipeline to `model/*.joblib` for the Streamlit app
6. Writes `test_data.csv` (the held-out 20%) for upload in the deployed app

> **Run order:** run every cell top to bottom. Cell 3 writes `test_data.csv` and the
> final training cell writes the `model/` folder. Both are required by `app.py`.

---
## 0. Environment and imports

In [ ]:
import json
import os
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score,
                             roc_curve)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib      :", joblib.__version__)

> **Pin these versions in `requirements.txt`.** Streamlit Community Cloud installs a fresh
> environment. If the scikit-learn version there differs from the one that pickled the
> models, `joblib.load` will either warn or fail outright.

---
## 1. Dataset

**Source:** UCI Machine Learning Repository - *Online Shoppers Purchasing Intention Dataset*
(Sakar, C.O., Polat, S.O., Katircioglu, M., Kastro, Y., 2018).

| Requirement | Assignment minimum | This dataset |
|---|---|---|
| Features | 12 | 17 |
| Instances | 500 | 12,330 |
| Task | Binary or multi-class | Binary |

In [ ]:
DATA_PATH = "online_shoppers_intention_2.csv"   # keep the CSV next to this notebook

df = pd.read_csv(DATA_PATH)
df["Weekend"] = df["Weekend"].astype(bool)
df["Revenue"] = df["Revenue"].astype(bool)

print("Shape:", df.shape)
df.head()

In [ ]:
# Structure, data types and completeness
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "n_missing": df.isna().sum(),
})
info

In [ ]:
# Target balance - the single most important fact about this dataset
counts = df["Revenue"].value_counts()
print(counts)
print("\nPositive class share: {:.2%}".format(df["Revenue"].mean()))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x="Revenue", data=df, ax=ax[0], palette="Blues_r")
ax[0].set_title("Class balance (Revenue)")
sns.boxplot(x="Revenue", y="PageValues", data=df, ax=ax[1], palette="Blues_r")
ax[1].set_title("PageValues by outcome")
ax[1].set_ylim(0, 60)
plt.tight_layout()
plt.show()

In [ ]:
# Numeric summary
df.describe().T.round(3)

### Feature dictionary

| Group | Columns | Notes |
|---|---|---|
| Page-count features | `Administrative`, `Informational`, `ProductRelated` | Pages of each type visited |
| Duration features | `*_Duration` | Seconds spent on each page type |
| Google Analytics metrics | `BounceRates`, `ExitRates`, `PageValues` | `PageValues` is the strongest single predictor |
| Seasonality | `SpecialDay`, `Month` | Proximity to a special day; calendar month |
| Technical | `OperatingSystems`, `Browser`, `Region`, `TrafficType` | Integer-coded **categorical** identifiers, not quantities |
| Visitor | `VisitorType`, `Weekend` | New / Returning / Other; weekend flag |
| **Target** | `Revenue` | `True` if the session ended in a transaction |

`OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as integers but carry no
ordinal meaning (Browser 8 is not "more" than Browser 2). They are one-hot encoded rather than
scaled, otherwise linear and distance-based models read a false ordering.

---
## 2. Feature / target split and column typing

In [ ]:
TARGET = "Revenue"

NUMERIC = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
CATEGORICAL = [
    "Month", "OperatingSystems", "Browser", "Region",
    "TrafficType", "VisitorType", "Weekend",
]

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)          # True -> 1, False -> 0

assert sorted(NUMERIC + CATEGORICAL) == sorted(X.columns), "Column list mismatch"
print("Features:", X.shape[1], "| numeric:", len(NUMERIC), "| categorical:", len(CATEGORICAL))

---
## 3. Train / test split

The split happens **before** any scaling or encoding. Fitting a `StandardScaler` on the full
dataset and splitting afterwards leaks test-set means and variances into training and inflates
every metric. Stratification keeps the 15% positive rate identical in both partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "positive rate {:.2%}".format(y_train.mean()))
print("Test :", X_test.shape, "positive rate {:.2%}".format(y_test.mean()))

# Held-out set written to disk - this is the file uploaded to the Streamlit app
test_df = X_test.copy()
test_df[TARGET] = y_test.map({1: True, 0: False}).values
test_df.to_csv("test_data.csv", index=False)
print("\nWrote test_data.csv:", test_df.shape)

---
## 4. Preprocessing pipeline

Every model is wrapped in a `Pipeline` so that `StandardScaler` and `OneHotEncoder` are fitted
on training folds only. `handle_unknown="ignore"` means a category present in an uploaded CSV
but absent from training does not crash the app.

`GaussianNB` cannot consume a sparse matrix, so it gets a dense variant of the same transformer.

In [ ]:
def build_preprocessor(dense: bool = False) -> ColumnTransformer:
    """Scale numeric columns, one-hot encode categorical columns."""
    return ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=not dense), CATEGORICAL),
    ])


# Inspect the encoded width
_prep = build_preprocessor().fit(X_train)
print("Encoded feature count:", _prep.transform(X_train).shape[1])

---
## 5. Models

Five classifiers, all on the identical feature set so the comparison is fair.

| # | Model | Key hyper-parameters | Rationale |
|---|---|---|---|
| 1 | Logistic Regression | `max_iter=2000` | Linear baseline |
| 2 | Decision Tree | `max_depth=8`, `min_samples_leaf=20` | Depth capped to stop memorisation |
| 3 | kNN | `n_neighbors=15`, `weights="distance"` | Needs the scaling step to be meaningful |
| 4 | Gaussian Naive Bayes | `var_smoothing=0.1` | See note below |
| 5 | Random Forest (ensemble) | `n_estimators=200`, `min_samples_leaf=5` | Bagged ensemble, required by the brief |

**Why `var_smoothing=0.1` on Naive Bayes.** One-hot columns are mostly zero, so their
within-class variance is near zero. With the default `var_smoothing=1e-9` the Gaussian
likelihood explodes on those columns and the model degenerates: it labels almost everything
positive (accuracy 0.27, precision 0.17). Raising the variance floor is a legitimate
hyper-parameter fix, not a workaround, and lifts accuracy to 0.79. This is worth stating in
the viva.

**On the model count.** The brief lists five models by name and the comparison table in the
brief has five rows, so five are implemented here. The phrase "all the 6 ML models" in the
brief appears to be a typo carried over from an earlier version.

In [ ]:
MODELS = {
    "Logistic Regression": (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), False),
    "Decision Tree": (
        DecisionTreeClassifier(max_depth=8, min_samples_leaf=20,
                               random_state=RANDOM_STATE), False),
    "kNN": (
        KNeighborsClassifier(n_neighbors=15, weights="distance"), False),
    "Naive Bayes": (
        GaussianNB(var_smoothing=0.1), True),          # True -> dense input
    "Random Forest (Ensemble)": (
        RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                               n_jobs=-1, random_state=RANDOM_STATE), False),
}

for name in MODELS:
    print("-", name)

---
## 6. Evaluation metrics

| Metric | Reads |
|---|---|
| Accuracy | Share of correct predictions. Misleading here: predicting "no purchase" for every session already scores 0.85 |
| AUC | Ranking quality across all thresholds, computed from `predict_proba`, not from hard labels |
| Precision | Of the sessions flagged as buyers, how many actually bought |
| Recall | Of the actual buyers, how many were caught |
| F1 | Harmonic mean of precision and recall |
| MCC | Correlation between predicted and true labels, balanced across all four confusion-matrix cells. The most trustworthy single number on an imbalanced target |

AUC must be computed on predicted **probabilities**. Passing hard labels silently returns
balanced accuracy instead and understates every model.

In [ ]:
def evaluate(name, pipe, X_te, y_te):
    """Fit-free scoring of one trained pipeline."""
    y_pred = pipe.predict(X_te)
    y_proba = pipe.predict_proba(X_te)[:, 1]      # probabilities, not labels
    return {
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_te, y_pred),
        "AUC": roc_auc_score(y_te, y_proba),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall": recall_score(y_te, y_pred, zero_division=0),
        "F1": f1_score(y_te, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_te, y_pred),
    }

---
## 7. Train, score and persist every model

In [ ]:
Path("model").mkdir(exist_ok=True)

results, fitted, curves = [], {}, {}

for name, (estimator, needs_dense) in MODELS.items():
    pipe = Pipeline([
        ("prep", build_preprocessor(dense=needs_dense)),
        ("clf", estimator),
    ])
    pipe.fit(X_train, y_train)

    results.append(evaluate(name, pipe, X_test, y_test))
    fitted[name] = pipe
    curves[name] = roc_curve(y_test, pipe.predict_proba(X_test)[:, 1])[:2]

    filename = (name.lower().replace(" ", "_")
                    .replace("(", "").replace(")", "") + ".joblib")
    joblib.dump(pipe, Path("model") / filename, compress=3)
    print("trained and saved:", name, "->", filename)

In [ ]:
results_df = (pd.DataFrame(results)
                .set_index("ML Model Name")
                .round(4))
results_df

In [ ]:
# Same table, highlighted - best value per metric
results_df.style.highlight_max(axis=0, color="#c8e6c9").format("{:.4f}")

In [ ]:
# Persist the comparison table for the README and the report
results_df.to_csv("model_metrics.csv")
print(results_df.to_markdown())

---
## 8. Diagnostics

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, pipe) in zip(axes.ravel(), fitted.items()):
    cm = confusion_matrix(y_test, pipe.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["No purchase", "Purchase"],
                yticklabels=["No purchase", "Purchase"])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
for ax in axes.ravel()[len(fitted):]:      # blank the unused panel
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ROC curves
plt.figure(figsize=(7, 6))
for name, (fpr, tpr) in curves.items():
    auc = results_df.loc[name, "AUC"]
    plt.plot(fpr, tpr, label="{} (AUC = {:.3f})".format(name, auc))
plt.plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves on the held-out test set")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Metric comparison bar chart
results_df[["Accuracy", "AUC", "F1", "MCC"]].plot(
    kind="bar", figsize=(11, 5), colormap="Blues_r", edgecolor="black")
plt.title("Model comparison on the held-out test set")
plt.ylabel("Score")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Which features drive the Random Forest
rf = fitted["Random Forest (Ensemble)"]
feature_names = rf.named_steps["prep"].get_feature_names_out()
importances = (pd.Series(rf.named_steps["clf"].feature_importances_, index=feature_names)
                 .sort_values(ascending=False)
                 .head(15))

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="Blues_r")
plt.title("Random Forest - top 15 features")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
importances.round(4)

In [ ]:
# Full classification report for the best model by MCC
best = results_df["MCC"].idxmax()
print("Best model by MCC:", best, "\n")
print(classification_report(y_test, fitted[best].predict(X_test),
                            target_names=["No purchase", "Purchase"], digits=4))

---
## 9. Observations

*(Numbers below come from the run above. Re-check them against your own output if you change
any hyper-parameter or the random seed.)*

| Model | Observation |
|---|---|
| **Logistic Regression** | Solid ranking (AUC 0.883) but the worst recall of the non-tree models at 0.359. The decision boundary is linear and the purchase signal is not: `PageValues` interacts with `ExitRates` and `Month` in ways a single hyperplane cannot capture. High precision (0.737), so what it flags is usually right, but it misses roughly two-thirds of buyers. |
| **Decision Tree** | Best F1 (0.627) and best MCC (0.572) of the five. Depth capped at 8 with `min_samples_leaf=20`, which is what keeps it from overfitting; an unconstrained tree drops sharply on the test set. Splits on `PageValues` capture the non-linear purchase threshold directly. |
| **kNN** | Weakest on AUC (0.829) and recall (0.301). After one-hot encoding the space is 75-dimensional and sparse, so Euclidean distances lose discrimination (curse of dimensionality). The 15% minority class is also routinely outvoted inside any 15-neighbour ball. |
| **Naive Bayes** | Lowest MCC (0.384) of the five. The conditional-independence assumption is plainly violated: `ProductRelated` and `ProductRelated_Duration` are near-collinear, as are `BounceRates` and `ExitRates`, so correlated evidence gets counted twice. It over-predicts purchases (precision 0.398, recall 0.634), the opposite failure mode to kNN. Required `var_smoothing=0.1` to stay usable on one-hot features. |
| **Random Forest (Ensemble)** | Best accuracy (0.898), best AUC (0.918) and best precision (0.784). Bagging removes the single tree's variance, but the default 0.5 threshold on an imbalanced target holds recall to 0.474, which is why its F1 sits below the single Decision Tree despite better ranking. Lowering the threshold or setting `class_weight="balanced"` trades precision for recall. |
| **Overall winner** | **Random Forest.** It leads on accuracy, AUC and precision, and the AUC advantage is threshold-independent, so it holds at any operating point. The Decision Tree is the counter-argument: it wins F1 and MCC, but only at the default 0.5 cut-off. Move the Random Forest threshold down to 0.35 and it reaches F1 0.663 and MCC 0.601, beating the tree on both, which is why the ranking metric is the better basis for the decision. |

### Cross-cutting points

1. **Accuracy is the wrong headline metric here.** A model that always predicts "no purchase"
   scores 0.845. Every model except Naive Bayes beats that by only a few points, while MCC
   spreads much wider: 0.384 for Naive Bayes against 0.572 for the Decision Tree.
2. **`PageValues` dominates.** It alone carries most of the Random Forest importance mass. It is
   a Google Analytics field derived from historic conversions on the pages visited, so it is
   close to a leading indicator of intent rather than an independent behavioural feature. Worth
   flagging as a mild target-leakage risk in a production setting.
3. **Recall is the binding constraint.** Every model finds the non-buyers easily and struggles
   on buyers. If the business objective is catching potential purchasers for an intervention,
   tune the decision threshold on the validation set rather than accepting 0.5.

In [ ]:
# Optional: 5-fold cross-validated AUC on the training set, as a stability check
for name, (estimator, needs_dense) in MODELS.items():
    pipe = Pipeline([("prep", build_preprocessor(dense=needs_dense)), ("clf", estimator)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="roc_auc", n_jobs=-1)
    print("{:<30} CV AUC = {:.4f} (+/- {:.4f})".format(name, scores.mean(), scores.std()))

---
## 10. Verify the saved artifacts

The Streamlit app loads these files. Confirm they round-trip before pushing to GitHub.

In [ ]:
for path in sorted(Path("model").glob("*.joblib")):
    pipe = joblib.load(path)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    print("{:<40} {:>7.1f} KB   reloaded accuracy = {:.4f}".format(
        path.name, path.stat().st_size / 1024, acc))

print("\ntest_data.csv rows:", len(pd.read_csv("test_data.csv")))

---
## 11. Deployment checklist

**Repository layout**

```
project-folder/
|-- app.py
|-- requirements.txt
|-- README.md
|-- test_data.csv
|-- ml_assignment2_online_shoppers.ipynb
|-- model/
    |-- logistic_regression.joblib
    |-- decision_tree.joblib
    |-- knn.joblib
    |-- naive_bayes.joblib
    |-- random_forest_ensemble.joblib
```

**Steps**

1. Run this notebook end to end on the BITS Virtual Lab. Take **one screenshot** showing the
   notebook running there with the comparison table visible.
2. `git init`, commit all files above, push to a **public** GitHub repository.
3. Go to https://streamlit.io/cloud, sign in with GitHub, click **New app**.
4. Select the repository, branch `main`, main file `app.py`, then **Deploy**.
5. Open the live URL, upload `test_data.csv`, switch models in the dropdown, confirm metrics
   and the confusion matrix render.
6. Assemble the submission PDF in this order: GitHub link, live Streamlit link, BITS Lab
   screenshot, full README content.

**Two failure modes to check before you submit**

- **Version drift.** `requirements.txt` must pin the same scikit-learn version that pickled the
  models (printed in cell 1). A mismatch breaks `joblib.load` on Streamlit Cloud.
- **Repository visibility.** A private repo deploys for you but the evaluator's link will 404.